### Importing data from the Calibration Log
I imagine we can also return a dataframe of values, but this had the needed info.

In [2]:
# Importing libraries
import numpy as np
import pandas as pd
import plotly.express as px
import geopandas as gpd


# Load the calibration log CSV file
calibration_log_df = pd.read_csv('calibration_log.csv')

# Select only the target_name and estimate columns
calibration_log_df = calibration_log_df[["target_name", "estimate"]]

# Filter rows where 'district' is in target_name, as we are only interested in congressional districts
calibration_log_df = calibration_log_df[calibration_log_df["target_name"].str.contains("age/district")]

# Extract the code following 'district_' and create a new column with the state and district code
calibration_log_df["geo_id"] = calibration_log_df["target_name"].str.extract(r"district_(\d+)[/]")

calibration_log_df.head(20)

,target_name,estimate,geo_id
36,age/district_0601/0-4,54168672.0,0601
37,age/district_0601/5-9,54168672.0,0601
38,age/district_0601/10-14,54168672.0,0601
39,age/district_0601/15-19,54168672.0,0601
40,age/district_0601/20-24,54168672.0,0601
41,age/district_0601/25-29,54168672.0,0601
42,age/district_0601/30-34,71097344.0,0601
43,age/district_0601/35-39,71097344.0,0601
44,age/district_0601/40-44,71097344.0,0601
45,age/district_0601/45-49,71097344.0,0601


Testing ground for checking what the shapefiles contain

In [3]:
shapefile = gpd.read_file('/Users/elenacura/Desktop/PolicyEngine/2024_Congressional_Districts')
filtered_shapefile = shapefile[shapefile['GEOID'].astype(str).str.startswith('06')]

# Check what you have
print(filtered_shapefile.head())
print(filtered_shapefile.columns)  # Look for district code column


         State District   Code              Name GEOID  \
21  California        1  CA-01  California's 1st  0601   
22  California        2  CA-02  California's 2nd  0602   
23  California        3  CA-03  California's 3rd  0603   
24  California        4  CA-04  California's 4th  0604   
25  California        5  CA-05  California's 5th  0605   

                                             geometry  
21  POLYGON ((-120.15191 41.99463, -120.15118 41.9...  
22  MULTIPOLYGON (((-124.21161 41.99846, -124.2116...  
23  POLYGON ((-116.47214 36.44656, -116.44523 36.4...  
24  POLYGON ((-121.6762 38.16677, -121.67665 38.16...  
25  POLYGON ((-121.10512 37.72144, -121.10512 37.7...  
Index(['State', 'District', 'Code', 'Name', 'GEOID', 'geometry'], dtype='object')


In [ ]:
# Use the geographic shapefile
shapefile = gpd.read_file('/Users/elenacura/Desktop/PolicyEngine/2024_Congressional_Districts')

# Aggregate and merge
district_totals = calibration_log_df.groupby('geo_id')['estimate'].sum().reset_index()
merged = shapefile.merge(district_totals, left_on='GEOID', right_on='geo_id', how='left')

# Create a proper choropleth
fig = px.choropleth_mapbox(merged, 
                           geojson=merged.geometry,
                           locations=merged.index,
                           color='estimate',
                           hover_data=['District', 'GEOID'],
                           mapbox_style="carto-positron",
                           zoom=5,
                           center={"lat": 37.5, "lon": -119.5},
                           opacity=0.7,
                           title="California Congressional Districts - Population Estimates"
                          )

fig.update_layout(height=800, width=1000)
fig.show()

In [ ]:
# Use the HEXCD file - it has the hexagons already made!
hexmap = gpd.read_file('/Users/elenacura/Desktop/PolicyEngine/HexCDv31')
ca_hexes = hexmap[hexmap['GEOID'].str.startswith('06')]

# Aggregate and merge
district_totals = calibration_log_df.groupby('geo_id')['estimate'].sum().reset_index()
merged = ca_hexes.merge(district_totals, left_on='GEOID', right_on='geo_id')

# Create hexagon map
fig = px.choropleth(merged,
                    geojson=merged.set_index('GEOID').geometry,
                    locations='GEOID',
                    color='estimate',
                    hover_data=['CDLABEL', 'estimate'],
                    color_continuous_scale='Viridis',
                    title="California Congressional Districts - Hexagon Cartogram"
                   )

fig.update_geos(
    visible=False,
    fitbounds="locations"
)

fig.update_layout(
    height=600, 
    width=800,
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()

## Max's request: Simulated Data

I gave all congressional districts a random value and plotted it both ways.

In [6]:
# Get all unique GEOIDs from the shapefile
all_geo_ids = shapefile['GEOID'].unique()

# Create a simulated dataset with random values for each GEOID
simulated_data = pd.DataFrame({
    'geo_id': all_geo_ids,
    'estimate': np.random.randint(10, 500001, size=len(all_geo_ids))
})
#simulated_data.head(20)

# Making the colorscale from the PE color scheme
PE_colorscale = [
    [0.0, "#616161"],     # DARK_GRAY
    [0.25, "#BDBDBD"],    # MEDIUM_LIGHT_GRAY
    [0.5, "#F7FDFC"],     # TEAL_LIGHT
    [0.75, "#227773"],    # TEAL_PRESSED
    [1.0, "#1d3e5e"]      # DARK_BLUE_HOVER
]


In [ ]:
# Use the geographic shapefile
shapefile = gpd.read_file('/Users/elenacura/Desktop/PolicyEngine/2024_Congressional_Districts')
merged = shapefile.merge(simulated_data, left_on='GEOID', right_on='geo_id', how='left')

# Create a proper choropleth
fig = px.choropleth_mapbox(merged, 
                           geojson=merged.geometry,
                           locations=merged.index,
                           color='estimate',
                           color_continuous_scale=PE_colorscale,
                           hover_data=['District', 'GEOID'],
                           mapbox_style="carto-positron",
                           zoom=3,
                           center={"lat": 37.5, "lon": -119.5},
                           opacity=0.7,
                           title="All Congressional Districts - (Simulated) Population Estimates"
                          )

fig.update_layout(height=800, width=1000)
fig.show()

In [ ]:
# Use the HEXCD file - it has the hexagons already made!
hexmap = gpd.read_file('/Users/elenacura/Desktop/PolicyEngine/HexCDv31')
# Aggregate and merge
merged = hexmap.merge(simulated_data, left_on='GEOID', right_on='geo_id')

# Create hexagon map
fig = px.choropleth(merged,
                    geojson=merged.set_index('GEOID').geometry,
                    locations='GEOID',
                    color='estimate',
                    hover_data=['CDLABEL', 'estimate'],
                    color_continuous_scale=PE_colorscale,
                    title="All Congressional Districts - Hexagon Cartogram"
                   )

fig.update_geos(
    visible=False,
    fitbounds="locations"
)

fig.update_layout(
    height=600, 
    width=800,
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()